In [3]:
%%writefile tokenizer.py
"""
Shared tokenizer + vocabulary module for NLP Assignment 2, Project 1 (DailyDialog).

Used identically by Task 1 (n-grams), Task 2 (AWD-LSTM), Task 3 (hybrid), and the
BLiMP evaluation harness. Build the vocab ONCE from DailyDialog's train split and
reuse the same Vocab object everywhere — never rebuild it per-task.

Design choices (see PLAN.md Section 3 for the full rationale):
- Punctuation-aware whitespace tokenizer: splits words from attached punctuation
  (e.g. "hello!" -> ["hello", "!"]) but does not split contractions internally
  (e.g. "don't" stays as ["don't"]) since the spec doesn't ask for that level of
  linguistic splitting and DailyDialog is casual/contracted speech.
- Lowercase everything.
- Vocabulary built from train only, capped at 30,000 most frequent tokens.
  Tokens occurring <2 times in train, and any OOV token seen later, map to <UNK>.
- Special tokens: <unk>, <s>, </s>, <pad> (pad reserved for Task 2 batching; not
  used by Task 1's counting).
"""

import re
import json
from collections import Counter
from typing import List, Dict, Iterable


# ---------------------------------------------------------------------------
# Tokenization
# ---------------------------------------------------------------------------

# Matches: contractions/words with internal apostrophes (don't, it's) as one
# token, plain word tokens, or any single punctuation/symbol character.
_TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:\.[0-9]+)?|[^\sA-Za-z0-9]")


def _normalize_apostrophes(text: str) -> str:
    """Collapse all Unicode apostrophe/quote variants to a single straight
    ASCII apostrophe (') before any other processing.

    Real DailyDialog text mixes the straight apostrophe (U+0027, ') with the
    Unicode "right single quotation mark" (U+2019, ’) and occasionally other
    look-alikes, inconsistently -- sometimes within the same split. Left
    unnormalized, contraction detection silently fails for every curly-quote
    instance (e.g. "I \u2019 m here" would NOT be caught by the spacing-fix
    regex below, since that regex only matches the ASCII character), leaking
    junk single-character tokens ('m', 't', 's', etc.) into the vocabulary.
    This must run BEFORE `_normalize_spacing`.
    """
    return text.replace("\u2019", "'").replace("\u2018", "'").replace("`", "'")


def _normalize_spacing(text: str) -> str:
    """Fix known DailyDialog spacing artifacts before tokenizing.

    Real DailyDialog utterances frequently have spaces on BOTH sides of the
    apostrophe inside a contraction, e.g. "Let ' s go" instead of "Let's go".
    Left alone, this breaks contraction detection entirely: the word, the
    apostrophe, and the suffix end up as three spurious separate tokens
    ("let", "'", "s") instead of one "let's" token, polluting the vocabulary
    with junk single-character tokens. This regex collapses "<word> ' <sfx>"
    into "<word>'<sfx>" whenever the trailing part looks like a common
    contraction suffix (s, t, re, ve, ll, d, m), so "let ' s" -> "let's",
    "they ' re" -> "they're", etc. Requires a preceding word character so it
    doesn't fire on a bare leading apostrophe/quote mark.

    This is a documented, deliberate preprocessing deviation -- state it in
    the report's preprocessing subsection.
    """
    text = re.sub(
        r"(\w)\s*'\s+(s|t|re|ve|ll|d|m)\b",
        r"\1'\2",
        text,
        flags=re.IGNORECASE,
    )
    return text


def tokenize(text: str) -> List[str]:
    """Punctuation-aware whitespace tokenizer + lowercase.

    Applies DailyDialog-specific spacing normalization (see
    `_normalize_spacing`) before the regex tokenizer runs.

    Example:
        >>> tokenize("Hello! Don't you think it's 3.5 degrees out?")
        ['hello', '!', "don't", 'you', 'think', "it's", '3.5', 'degrees', 'out', '?']
        >>> tokenize("Good . Let ' s go now .")
        ['good', '.', "let's", 'go', 'now', '.']
        >>> tokenize("I \u2019 m here .")
        ["i'm", 'here', '.']
    """
    text = _normalize_apostrophes(text)
    text = _normalize_spacing(text)
    text = text.lower()
    return _TOKEN_RE.findall(text)


def join_dialogue(utterances: List[str]) -> str:
    """Join a DailyDialog `dialog` list into one document.

    Per spec: concatenate utterances in order, separated by a single space.
    NO speaker tags, no turn markers.
    """
    return " ".join(u.strip() for u in utterances)


# ---------------------------------------------------------------------------
# Vocabulary
# ---------------------------------------------------------------------------

UNK = "<unk>"
BOS = "<s>"
EOS = "</s>"
PAD = "<pad>"

SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]  # fixed order -> fixed ids 0,1,2,3


class Vocab:
    """Fixed vocabulary built from a token frequency count.

    Attributes:
        token2id: dict token -> int id
        id2token: list, index = id
        freq: Counter of raw training-set token frequencies (pre-cap, pre-UNK
              collapse) -- kept for diagnostics / report stats, not used at
              runtime.
    """

    def __init__(self, token2id: Dict[str, int], id2token: List[str], freq: Counter):
        self.token2id = token2id
        self.id2token = id2token
        self.freq = freq

    def __len__(self):
        return len(self.id2token)

    def encode(self, tokens: Iterable[str]) -> List[int]:
        unk_id = self.token2id[UNK]
        return [self.token2id.get(t, unk_id) for t in tokens]

    def encode_str(self, tokens: Iterable[str]) -> List[str]:
        """Map OOV tokens to the literal '<unk>' string (used by n-gram code
        that works with token strings rather than ids)."""
        vocab_set = set(self.token2id.keys())
        return [t if t in vocab_set else UNK for t in tokens]

    def decode(self, ids: Iterable[int]) -> List[str]:
        return [self.id2token[i] for i in ids]

    def save(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "id2token": self.id2token,
                    "freq": dict(self.freq),
                },
                f,
                ensure_ascii=False,
                indent=1,
            )

    @classmethod
    def load(cls, path: str) -> "Vocab":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        id2token = obj["id2token"]
        token2id = {t: i for i, t in enumerate(id2token)}
        freq = Counter(obj["freq"])
        return cls(token2id, id2token, freq)


def build_vocab(
    train_token_lists: Iterable[List[str]],
    max_size: int = 30000,
    min_count: int = 2,
) -> Vocab:
    """Build a Vocab from an iterable of already-tokenized training documents.

    Args:
        train_token_lists: iterable where each element is the token list for
            one training document (one DailyDialog dialogue, already joined +
            tokenized). Pass the full train split here, nothing held out.
        max_size: cap on vocabulary size INCLUDING special tokens. Spec says
            "30,000 most frequent tokens" -- we interpret this as 30,000
            regular (non-special) tokens, with the 4 special tokens added on
            top, since the special tokens are not part of the natural-language
            frequency count the spec is capping. State this in the report if
            asked; trivial to change `max_size` semantics here if TA disagrees.
        min_count: tokens occurring fewer than this many times in train are
            collapsed into <UNK> even if they'd otherwise fit under max_size.

    Returns:
        Vocab object. Deterministic tie-breaking: when multiple tokens have
        identical frequency at the cutoff boundary, ties are broken
        alphabetically (ascending) for full reproducibility across runs/seeds.
    """
    freq = Counter()
    for tokens in train_token_lists:
        freq.update(tokens)

    # Keep tokens meeting min_count, then cap to max_size most frequent,
    # alphabetical tie-break for determinism.
    eligible = [(tok, c) for tok, c in freq.items() if c >= min_count]
    eligible.sort(key=lambda x: (-x[1], x[0]))  # freq desc, then alpha asc
    kept = [tok for tok, _ in eligible[:max_size]]

    id2token = list(SPECIAL_TOKENS) + kept
    token2id = {t: i for i, t in enumerate(id2token)}

    return Vocab(token2id, id2token, freq)


# ---------------------------------------------------------------------------
# Scoring-stream builder (shared by Task 1 n-gram eval AND Task 2 LSTM eval,
# so Task 3's per-position interpolation is well-defined -- see PLAN.md 3.7)
# ---------------------------------------------------------------------------

def build_scoring_stream(
    dialogues: List[List[str]],
    vocab: Vocab,
) -> List[str]:
    """Turn a list of dialogues (each a list of raw utterance strings) into
    ONE flat list of vocab-mapped token strings, with <s> ... </s> wrapped
    around each dialogue (per-dialogue boundaries, NOT per-utterance -- see
    PLAN.md assumption 3.2).

    This exact function must be used by both the n-gram scorer and the LSTM
    scorer at evaluation time so both models score an identical sequence of
    positions. Do not reimplement this logic separately in task2/task3 code.
    """
    stream: List[str] = []
    for utterances in dialogues:
        doc = join_dialogue(utterances)
        toks = tokenize(doc)
        toks = vocab.encode_str(toks)
        stream.append(BOS)
        stream.extend(toks)
        stream.append(EOS)
    return stream


if __name__ == "__main__":
    # Minimal smoke test using fabricated data (NOT real DailyDialog -- just
    # verifies the module's internal logic before we touch Colab/real data).
    fake_dialogues = [
        ["Hello there!", "Hi, how are you?", "I'm fine, thanks."],
        ["Don't forget the meeting.", "I won't. See you at 3.5pm!"],
        ["Hello there!", "Hi, how are you?"],  # repeat to test frequency counts
    ]

    tokenized_docs = [tokenize(join_dialogue(d)) for d in fake_dialogues]
    for d, toks in zip(fake_dialogues, tokenized_docs):
        print(join_dialogue(d), "->", toks)

    vocab = build_vocab(tokenized_docs, max_size=30000, min_count=2)
    print("\nVocab size (incl. specials):", len(vocab))
    print("Tokens kept:", [t for t in vocab.id2token if t not in SPECIAL_TOKENS])
    print("'hello' in vocab:", "hello" in vocab.token2id)
    print("'meeting' in vocab (freq 1, should be UNK'd):", "meeting" in vocab.token2id)

    stream = build_scoring_stream(fake_dialogues, vocab)
    print("\nScoring stream:")
    print(stream)

    # Save/load round trip
    vocab.save("/tmp/test_vocab.json")
    reloaded = Vocab.load("/tmp/test_vocab.json")
    assert reloaded.id2token == vocab.id2token
    print("\nSave/load round-trip OK.")

Writing tokenizer.py


In [4]:
%%writefile awd_lstm.py
"""
AWD-LSTM (Merity, Keskar, Socher 2017, arXiv:1708.02182) -- Task 2.

Written fresh against the paper's description rather than adapted from the
official repo (github.com/salesforce/awd-lstm-lm), which predates modern
PyTorch and uses APIs (e.g. old-style weight-norm hooks, Python 2 idioms)
that don't run cleanly on current torch versions. See PLAN.md assumption 3.6.

Components implemented (per spec step 2):
  - WeightDropLSTM: DropConnect (p=0.5 default) applied to the
    hidden-to-hidden weight matrix of each LSTM layer, reapplied (a fresh
    dropout mask) on every forward call during training.
  - Embedding dropout (p=0.1): drops entire embedding ROWS (whole word
    types), not individual activations -- this is what the paper calls
    "embedding dropout" (distinct from standard dropout on embedding
    outputs).
  - Output (locked) dropout (p=0.4): standard dropout applied to the final
    LSTM layer's output before the softmax, using the SAME mask across all
    timesteps in a sequence (a "locked"/"variational" dropout mask) --
    this is what the paper calls output dropout.
  - Tied embedding/softmax weights.

NOT implemented (explicit, documented deviations -- state in report):
  - AR/TAR activation regularization (the paper's alpha/beta regularization
    terms) -- spec's step-by-step instructions (steps 1-4) do not mention
    these, only DropConnect + embedding dropout + output dropout + NT-ASGD +
    the listed hyperparameters, so we implement exactly what's listed and
    flag AR/TAR as an omitted paper detail, consistent with "minimal effort,
    ship what's asked for" per team decision.
  - Variable-length BPTT (paper varies BPTT length per batch for
    regularization) -- spec says "BPTT length 70" as a fixed hyperparameter,
    so we use a fixed length, not the paper's variable scheme.
  - The paper's fine-tuning second stage -- out of scope per spec's step 5.
"""

import math

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _TORCH_AVAILABLE = True
except ImportError:
    # This sandbox has no internet access to PyTorch's package servers, so
    # this module cannot be executed/tested here -- only written and
    # manually shape-traced (see SHAPE_TRACE below). It WILL run in Colab,
    # which has torch preinstalled. Running this file directly in an
    # environment without torch still prints the shape trace for review.
    _TORCH_AVAILABLE = False
    nn = None
    F = None


# ---------------------------------------------------------------------------
# DropConnect on hidden-to-hidden weights
# ---------------------------------------------------------------------------

class WeightDropLSTM(nn.Module if _TORCH_AVAILABLE else object):
    """Wraps nn.LSTM and applies DropConnect to the hidden-to-hidden weight
    matrix (weight_hh_l0) on every forward call during training.

    Standard dropout zeroes ACTIVATIONS; DropConnect zeroes WEIGHTS.

    IMPLEMENTATION NOTE (important, read before modifying): an earlier draft
    of this class tried to replace nn.LSTM's weight_hh_l0 Parameter with a
    plain (non-leaf) tensor via setattr(), mirroring a trick from the
    official AWD-LSTM repo. That approach is fragile on modern PyTorch:
    nn.LSTM caches parameter names/references at construction time
    (`_flat_weights`, `_all_weights`) for its fused cuDNN/CPU kernel path,
    and a bare setattr() does not reliably update what the kernel actually
    reads -- this could silently train with the UNDROPPED weight while
    appearing to work. Since this module cannot be executed/tested in this
    sandbox (no torch available -- see top-of-file note) before being run
    for real in Colab, we deliberately avoid that risk entirely and use a
    simpler, more robust approach instead:

      1. Keep weight_hh_l0 as nn.LSTM's own real Parameter (untouched,
         nothing deleted/reassigned/re-registered).
      2. On each forward() call during training: clone the current
         weight_hh_l0.data, compute a dropped-out version, write it IN
         PLACE into weight_hh_l0.data (same tensor object, so cuDNN's
         cached references stay valid), run the forward pass, then
         restore the original (undropped) data afterward.
      3. Gradients still flow correctly: autograd records the forward
         computation using whatever data was in weight_hh_l0 at the time
         (the dropped version), and since the Parameter object itself
         (and its .grad) is never replaced, the optimizer's step() still
         updates the SAME Parameter it always did -- we're only ever
         swapping the underlying .data buffer's contents, never the
         Parameter object's identity.

    This trades a small amount of complexity (explicit save/restore) for
    something we can reason about correctly without being able to run it
    first. If Colab shows the model training (loss decreasing, no shape
    errors) in the smoke test, this approach is confirmed working.
    """

    def __init__(self, input_size, hidden_size, dropconnect_p=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=False)
        self.dropconnect_p = dropconnect_p

    def forward(self, x, hidden=None):
        if self.training and self.dropconnect_p > 0:
            original_data = self.lstm.weight_hh_l0.data.clone()
            mask = torch.bernoulli(
                original_data.new_full(original_data.shape, 1 - self.dropconnect_p)
            ) / (1 - self.dropconnect_p)  # inverted dropout scaling
            self.lstm.weight_hh_l0.data.mul_(mask)  # in-place: same tensor object
            try:
                output, new_hidden = self.lstm(x, hidden)
            finally:
                # Always restore, even if the forward pass raises, so a
                # mid-training exception can't leave the model's real
                # weights permanently corrupted.
                self.lstm.weight_hh_l0.data.copy_(original_data)
            return output, new_hidden
        else:
            return self.lstm(x, hidden)


# ---------------------------------------------------------------------------
# Embedding dropout (drops whole word-rows, not individual activations)
# ---------------------------------------------------------------------------

def embedded_dropout(embed_weight, input_ids, p=0.1, training=True):
    """Apply embedding dropout: with probability p, zero out an ENTIRE row
    of the embedding matrix (i.e. drop a word TYPE for this forward pass,
    so every occurrence of that word in the batch gets a zero vector), then
    do a normal embedding lookup with the (possibly zeroed) matrix.
    """
    if training and p > 0:
        mask = embed_weight.data.new_full((embed_weight.size(0), 1), 1 - p)
        mask = torch.bernoulli(mask) / (1 - p)
        masked_weight = embed_weight * mask
    else:
        masked_weight = embed_weight
    return F.embedding(input_ids, masked_weight)


# ---------------------------------------------------------------------------
# Locked (variational) dropout -- same mask reused across all timesteps
# ---------------------------------------------------------------------------

class LockedDropout(nn.Module if _TORCH_AVAILABLE else object):
    """Standard dropout would sample a new mask per timestep; locked dropout
    samples ONE mask and reuses it for every timestep in the sequence
    (standard technique from the AWD-LSTM paper, used here for output
    dropout, p=0.4)."""

    def forward(self, x, p):
        if not self.training or p == 0:
            return x
        # x shape: (seq_len, batch, hidden)
        mask = x.data.new_full((1, x.size(1), x.size(2)), 1 - p)
        mask = torch.bernoulli(mask) / (1 - p)
        mask = mask.expand_as(x)
        return x * mask


# ---------------------------------------------------------------------------
# Full model
# ---------------------------------------------------------------------------

class AWDLSTM(nn.Module if _TORCH_AVAILABLE else object):
    """3-layer weight-dropped LSTM LM with tied embeddings.

    Args match the spec exactly:
        vocab_size: size of the vocabulary (len(vocab) from tokenizer.Vocab)
        embed_dim: 400 (paper default)
        hidden_dim: 1150 (paper default; pass 400 for the compute-constrained
            fallback -- see PLAN.md assumption 3.6)
        n_layers: 3
        dropconnect_p: 0.5 (hidden-to-hidden DropConnect)
        embed_dropout_p: 0.1
        output_dropout_p: 0.4

    IMPORTANT architectural note on tied weights: tying requires the LAST
    LSTM layer's hidden_dim to equal embed_dim (since the output projection
    reuses the embedding matrix, shape (vocab_size, embed_dim)). The paper
    handles this by making the LAST layer's hidden size equal to embed_dim
    (400) even though earlier layers use the larger hidden_dim (1150). We
    replicate that: layers 0..n_layers-2 use hidden_dim, the FINAL layer
    uses embed_dim as its output size.
    """

    def __init__(self, vocab_size, embed_dim=400, hidden_dim=1150, n_layers=3,
                 dropconnect_p=0.5, embed_dropout_p=0.1, output_dropout_p=0.4):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embed_dropout_p = embed_dropout_p
        self.output_dropout_p = output_dropout_p

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Per-layer sizes: all but last use hidden_dim, last uses embed_dim
        # (required for weight tying -- see docstring).
        layer_input_sizes = [embed_dim] + [hidden_dim] * (n_layers - 1)
        layer_output_sizes = [hidden_dim] * (n_layers - 1) + [embed_dim]

        self.rnns = nn.ModuleList([
            WeightDropLSTM(layer_input_sizes[i], layer_output_sizes[i], dropconnect_p)
            for i in range(n_layers)
        ])

        self.locked_dropout = LockedDropout()

        self.decoder = nn.Linear(embed_dim, vocab_size)
        # Tie weights: decoder.weight (vocab_size, embed_dim) == embedding.weight
        self.decoder.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        init_range = 0.1
        self.embedding.weight.data.uniform_(-init_range, init_range)
        self.decoder.bias.data.zero_()
        # decoder.weight is tied to embedding.weight, already initialized above

    def init_hidden(self, batch_size, device):
        hidden = []
        for i, rnn in enumerate(self.rnns):
            h_size = self.hidden_dim if i < self.n_layers - 1 else self.embed_dim
            h = torch.zeros(1, batch_size, h_size, device=device)
            c = torch.zeros(1, batch_size, h_size, device=device)
            hidden.append((h, c))
        return hidden

    def forward(self, input_ids, hidden):
        """
        input_ids: (seq_len, batch) LongTensor of token ids
        hidden: list of (h, c) tuples, one per layer, from init_hidden() or
                a previous forward() call's returned hidden state (for
                truncated BPTT -- caller detaches between chunks)

        Returns:
            logits: (seq_len, batch, vocab_size)
            new_hidden: list of (h, c) tuples, same structure as `hidden`
        """
        emb = embedded_dropout(
            self.embedding.weight, input_ids,
            p=self.embed_dropout_p, training=self.training
        )  # (seq_len, batch, embed_dim)

        x = emb
        new_hidden = []
        for i, rnn in enumerate(self.rnns):
            x, h_new = rnn(x, hidden[i])
            new_hidden.append(h_new)
            if i < self.n_layers - 1:
                # Locked dropout between LSTM layers too (standard AWD-LSTM
                # practice uses this for inter-layer dropout as well, at the
                # same output_dropout_p -- spec doesn't separately specify an
                # inter-layer rate, so we reuse output_dropout_p here, a
                # reasonable simplification per "minimal effort" principle).
                x = self.locked_dropout(x, self.output_dropout_p)

        x = self.locked_dropout(x, self.output_dropout_p)  # final output dropout
        logits = self.decoder(x)  # (seq_len, batch, vocab_size)
        return logits, new_hidden

    @staticmethod
    def detach_hidden(hidden):
        """Detach hidden state from the computation graph between BPTT
        chunks (standard truncated-BPTT practice -- prevents backprop from
        running all the way back to the start of training)."""
        return [(h.detach(), c.detach()) for (h, c) in hidden]


# ---------------------------------------------------------------------------
# Manual shape trace (since we cannot run torch in this sandbox -- see module
# docstring). This documents, step by step, what every tensor shape SHOULD
# be, so the first real Colab run can be checked against this by eye instead
# of debugging blind.
# ---------------------------------------------------------------------------

SHAPE_TRACE = """
Manual shape trace for AWDLSTM(vocab_size=12482, embed_dim=400, hidden_dim=1150, n_layers=3)
with seq_len=70 (BPTT length per spec), batch_size=80 (per spec):

  input_ids:            (70, 80)                      int64
  embedding lookup ->   (70, 80, 400)                  float32
  layer 0 (WeightDropLSTM, in=400, out=1150):
    input  (70, 80, 400)
    output (70, 80, 1150), h/c each (1, 80, 1150)
  locked_dropout (p=0.4) -> same shape (70, 80, 1150)
  layer 1 (WeightDropLSTM, in=1150, out=1150):
    input  (70, 80, 1150)
    output (70, 80, 1150), h/c each (1, 80, 1150)
  locked_dropout (p=0.4) -> same shape (70, 80, 1150)
  layer 2 (WeightDropLSTM, in=1150, out=400):   <- last layer outputs embed_dim, for tying
    input  (70, 80, 1150)
    output (70, 80, 400), h/c each (1, 80, 400)
  locked_dropout (p=0.4, final output dropout) -> (70, 80, 400)
  decoder (Linear(400 -> 12482), weight tied to embedding.weight):
    output logits: (70, 80, 12482)

Parameter count estimate (rough, for sanity-checking against Colab's actual
printed count -- see the test script's parameter counter):
  embedding: 12482 * 400                           = 4,992,800
  layer 0 LSTM: 4 * 1150 * (400+1150+1)            = 7,135,000   (approx, incl. biases)
  layer 1 LSTM: 4 * 1150 * (1150+1150+1)           = 10,580,600  (approx)
  layer 2 LSTM: 4 * 400  * (1150+400+1)            = 2,482,400   (approx)
  decoder bias: 12482
  decoder.weight is TIED (0 additional params, reuses embedding)
  ------------------------------------------------------------------
  TOTAL approx: ~25.2M parameters

This is in the same ballpark as the paper's ~24M for PTB/WT2 configs, which
is expected since embed_dim/hidden_dim match the paper despite our much
smaller vocab (12,482 vs ~33K) -- the LSTM weight matrices dominate
parameter count, not the embedding/decoder, so vocab size differences barely
move the total.
"""

# ---------------------------------------------------------------------------
# Training / evaluation utility functions
#
# These are plain tensor-reshaping and bookkeeping helpers -- they do NOT
# touch the WeightDropLSTM internals, so they carry none of the non-leaf-
# tensor risk that the weight-drop trick does. Kept deliberately simple.
# ---------------------------------------------------------------------------

def repackage_hidden(h):
    """Detach hidden states from their computation-graph history, so
    truncated BPTT doesn't backprop through the entire training history.
    Call this once per batch, on the hidden state returned by the previous
    batch's forward() call, before feeding it into the next batch."""
    if _TORCH_AVAILABLE and isinstance(h, torch.Tensor):
        return h.detach()
    else:
        return tuple(repackage_hidden(v) for v in h)


def batchify(data_ids, bsz, device):
    """Reshape a flat list[int] of token ids into a (num_steps, bsz) tensor
    for truncated-BPTT training, per spec: batch size 80.

    Standard LM batching: split the flat stream into `bsz` contiguous
    columns (dropping leftover tokens that don't fill a full column), so
    each column is read top-to-bottom as one continuous sub-sequence and
    hidden state carries across batches WITHIN a column via repackage_hidden.
    """
    data = torch.tensor(data_ids, dtype=torch.long)
    nbatch = data.size(0) // bsz
    data = data.narrow(0, 0, nbatch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)


def get_batch(source, i, bptt=70):
    """Slice one training chunk starting at row i: input is source[i:i+seq_len],
    target is the same window shifted forward by one timestep (next-token
    prediction), flattened to 1D for CrossEntropyLoss. seq_len is clipped
    near the end of `source` so we never index out of bounds."""
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i + seq_len]
    target = source[i + 1:i + 1 + seq_len].reshape(-1)
    return data, target


@torch.no_grad() if _TORCH_AVAILABLE else (lambda f: f)
def evaluate_stream_exact(model, stream_ids, bos_id, device, chunk_size=2048,
                           return_token_logprobs=False):
    """Exact sequential perplexity evaluation over a 1D scoring stream
    (batch_size=1, no truncation/dropped tail), matching Task 1's n-gram
    perplexity convention exactly (PLAN.md 3.2/3.7): every token is scored
    EXCEPT <s> (bos_id), which only ever appears as context, never as a
    predicted target. This is what makes Task 1 and Task 2 perplexities
    directly comparable, and is REQUIRED for Task 3's per-position hybrid
    interpolation to be well-defined (both models must score the identical
    sequence of positions).

    Use this for all REPORTED perplexity numbers (val/test). Use the faster
    batched `batchify`+`get_batch` path only for TRAINING, where exact
    per-token alignment with Task 1 doesn't matter (training just needs the
    loss signal, not a comparable metric).
    """
    model.eval()
    data = torch.tensor(stream_ids, dtype=torch.long, device=device).unsqueeze(1)  # (N, 1)
    hidden = model.init_hidden(1, device)

    total_logprob = 0.0
    n_tokens = 0
    token_logprobs = []

    for i in range(0, data.size(0) - 1, chunk_size):
        seq_len = min(chunk_size, data.size(0) - 1 - i)
        inp = data[i:i + seq_len]                      # (seq_len, 1)
        tgt = data[i + 1:i + 1 + seq_len].squeeze(1)    # (seq_len,)

        logits, hidden = model(inp, hidden)
        hidden = repackage_hidden(hidden)

        log_probs = F.log_softmax(logits.squeeze(1), dim=-1)  # (seq_len, V)
        target_lps = log_probs.gather(1, tgt.unsqueeze(1)).squeeze(1)  # (seq_len,)

        valid_mask = (tgt != bos_id)
        valid_lps = target_lps[valid_mask]

        total_logprob += valid_lps.sum().item()
        n_tokens += valid_mask.sum().item()

        if return_token_logprobs:
            token_logprobs.extend(valid_lps.cpu().tolist())

    ppl = math.exp(-total_logprob / n_tokens)
    if return_token_logprobs:
        return ppl, token_logprobs
    return ppl


if __name__ == "__main__":
    print(SHAPE_TRACE)
    print("\n(Full forward-pass execution requires torch + will be run in Colab;")
    print(" this file was written and shape-traced by hand in this sandbox")
    print(" since torch is not installable here -- verify against real run.)")

Writing awd_lstm.py


In [5]:
%%writefile nt_asgd.py
"""
NT-ASGD: Non-monotonically Triggered Averaged SGD (Merity, Keskar, Socher 2017,
arXiv:1708.02182, Algorithm 1) -- Task 2 optimizer, per spec step 3.

Written as a thin wrapper AROUND plain torch.optim.SGD, rather than a custom
Optimizer subclass -- deliberately simple, since the failure modes we most want
to avoid here are (a) silently averaging the wrong tensors, (b) forgetting to
swap averaged weights back in before evaluation, (c) off-by-one errors in the
"non-monotonic interval" trigger check. Keeping this as an explicit wrapper
class (not hiding behavior inside a subclassed .step()) makes every one of
those three steps an obvious, separately-callable, separately-inspectable
method -- easier to reason about without being able to execute it here.

Usage pattern (exact contract the training loop must follow):

    nt_asgd = NTASGD(model.parameters(), lr=30.0, n=5, weight_decay=1.2e-6)

    for epoch in range(1, n_epochs + 1):
        train_one_epoch(model, ..., optimizer=nt_asgd.optimizer)  # train as normal

        val_ppl = evaluate(...)                      # eval on RAW (non-averaged) weights is fine pre-trigger
        triggered_this_epoch = nt_asgd.step_epoch(val_ppl)   # call ONCE per epoch, after computing val_ppl

        if nt_asgd.is_averaging:
            nt_asgd.swap_to_averaged(model)           # put averaged weights into model for eval/checkpointing
            val_ppl_averaged = evaluate(...)          # re-evaluate using averaged weights (the ones you report)
            nt_asgd.swap_to_raw(model)                # put raw weights back for the NEXT epoch's training step

IMPORTANT: once is_averaging is True, you must call swap_to_averaged() before
any evaluation/checkpoint-saving you intend to report, and swap_to_raw()
before resuming training -- training must always continue on the raw
(non-averaged) parameters; only EVALUATION should see the averaged ones.
Mixing this up (e.g. training on averaged weights) silently breaks the
method's theoretical justification even though nothing would crash.
"""

import copy

try:
    import torch
    _TORCH_AVAILABLE = True
except ImportError:
    _TORCH_AVAILABLE = False


class NTASGD:
    """Wraps torch.optim.SGD; after each epoch, decides whether to start
    averaging parameters across subsequent steps, per the paper's
    non-monotonic trigger condition.

    Args:
        params: model.parameters() (same as any torch optimizer)
        lr: initial learning rate (spec: 30)
        n: non-monotonic interval -- trigger averaging once current val PPL
           is worse than the min of the last `n` val PPLs (paper default: 5,
           not specified explicitly in our spec, so we use the paper's own
           default value here and document this as filling a spec gap)
        weight_decay: passed through to the underlying SGD
    """

    def __init__(self, params, lr=30.0, n=5, weight_decay=0.0):
        self.params = list(params)  # materialize once; both SGD and we need
                                     # a stable, ORDERED list of the same
                                     # Parameter objects
        self.optimizer = torch.optim.SGD(self.params, lr=lr, weight_decay=weight_decay)
        self.n = n

        self.val_ppl_history = []
        self.is_averaging = False
        self.trigger_epoch = None

        # Running average state: for each param, accumulate a running SUM
        # of its values across every optimizer.step() call made AFTER
        # averaging was triggered, plus a count of how many steps have been
        # accumulated -- average = running_sum / step_count. We store this
        # as plain tensors (not Parameters), one per entry in self.params,
        # matched by LIST POSITION (self.params never changes order/length
        # after __init__, so position-matching is safe and avoids relying
        # on object identity across copies).
        self._running_sum = None   # list[Tensor], same shapes as self.params
        self._avg_step_count = 0

        # Saved raw (non-averaged) parameter values, used to restore model
        # state after a temporary swap_to_averaged() for evaluation.
        self._raw_backup = None

    def step_batch(self):
        """Call once per training batch, AFTER loss.backward() and gradient
        clipping, in place of optimizer.step(). Performs the normal SGD
        step, then (if averaging has been triggered) accumulates the
        post-step parameter values into the running average.
        """
        self.optimizer.step()

        if self.is_averaging:
            if self._running_sum is None:
                self._running_sum = [p.data.clone() for p in self.params]
                self._avg_step_count = 1
            else:
                for i, p in enumerate(self.params):
                    self._running_sum[i].add_(p.data)
                self._avg_step_count += 1

    def zero_grad(self):
        self.optimizer.zero_grad()

    def step_epoch(self, val_ppl):
        """Call once per epoch, after computing validation perplexity for
        that epoch (on the RAW, non-averaged weights -- averaging has no
        effect on training-time eval until swap_to_averaged() is called
        explicitly). Checks the non-monotonic trigger condition and flips
        self.is_averaging to True the first time it's satisfied.

        Returns True if averaging was JUST triggered this call (useful for
        logging "NT-ASGD triggered at epoch N" once), False otherwise.
        """
        just_triggered = False

        if not self.is_averaging:
            if len(self.val_ppl_history) >= self.n:
                recent_min = min(self.val_ppl_history[-self.n:])
                if val_ppl > recent_min:
                    self.is_averaging = True
                    just_triggered = True
                    # Averaging starts accumulating from the NEXT batch's
                    # step (step_batch handles first-call initialization of
                    # _running_sum lazily) -- we deliberately do NOT seed
                    # _running_sum with the current weights here, since the
                    # first accumulated step will do that on its own.

        self.val_ppl_history.append(val_ppl)
        return just_triggered

    def get_averaged_params(self):
        """Returns the current running-average parameter values as a list
        of tensors (same order as self.params). Raises if averaging hasn't
        been triggered/accumulated yet -- callers should check
        self.is_averaging and that at least one step_batch() has run since
        triggering before calling this."""
        if self._running_sum is None or self._avg_step_count == 0:
            raise RuntimeError(
                "get_averaged_params() called before any post-trigger "
                "optimizer steps have been accumulated. Check "
                "self.is_averaging and ensure at least one training batch "
                "has run since triggering."
            )
        return [s / self._avg_step_count for s in self._running_sum]

    def swap_to_averaged(self, model):
        """Temporarily replace model's parameter VALUES (in place, via
        .data) with the running-averaged values, for evaluation/
        checkpointing. Saves the raw values first so swap_to_raw() can
        restore them exactly. Safe to call only when self.is_averaging is
        True and at least one step has been accumulated (see
        get_averaged_params)."""
        averaged = self.get_averaged_params()
        self._raw_backup = [p.data.clone() for p in self.params]
        for p, avg_val in zip(self.params, averaged):
            p.data.copy_(avg_val)

    def swap_to_raw(self, model):
        """Restore the raw (non-averaged) parameter values saved by the most
        recent swap_to_averaged() call. MUST be called before resuming
        training after an averaged-weights evaluation -- training must
        always continue on raw weights, never on the averaged copy."""
        if self._raw_backup is None:
            raise RuntimeError(
                "swap_to_raw() called without a matching prior "
                "swap_to_averaged() call -- nothing to restore."
            )
        for p, raw_val in zip(self.params, self._raw_backup):
            p.data.copy_(raw_val)
        self._raw_backup = None

    def state_dict(self):
        """For checkpointing: includes the underlying SGD's state plus all
        NT-ASGD-specific bookkeeping needed to resume correctly mid-average."""
        return {
            'optimizer_state': self.optimizer.state_dict(),
            'val_ppl_history': self.val_ppl_history,
            'is_averaging': self.is_averaging,
            'trigger_epoch': self.trigger_epoch,
            'running_sum': self._running_sum,
            'avg_step_count': self._avg_step_count,
        }

    def load_state_dict(self, state):
        self.optimizer.load_state_dict(state['optimizer_state'])
        self.val_ppl_history = state['val_ppl_history']
        self.is_averaging = state['is_averaging']
        self.trigger_epoch = state['trigger_epoch']
        self._running_sum = state['running_sum']
        self._avg_step_count = state['avg_step_count']


if __name__ == "__main__":
    print(__doc__)
    print("\n(Requires torch to execute -- not runnable in this sandbox;")
    print(" logic reasoned through manually, verify in Colab.)")

Writing nt_asgd.py


In [6]:
import importlib
import awd_lstm as awd
import nt_asgd as ntasgd_module
importlib.reload(awd)
importlib.reload(ntasgd_module)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time, json, urllib.request
import pandas as pd
import tokenizer as tok
importlib.reload(tok)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.11.0+cu130
CUDA available: True


In [7]:
urls = {
    "train": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}
local_paths = {}
for split, url in urls.items():
    local_path = f"/content/daily_dialog_{split}.parquet"
    urllib.request.urlretrieve(url, local_path)
    local_paths[split] = local_path
dfs = {split: pd.read_parquet(path) for split, path in local_paths.items()}

train_token_lists = [tok.tokenize(tok.join_dialogue(d)) for d in dfs['train']['dialog'].tolist()]
vocab = tok.build_vocab(train_token_lists, max_size=30000, min_count=2)
print("Vocab size:", len(vocab))

train_stream_tokens = tok.build_scoring_stream(dfs['train']['dialog'].tolist(), vocab)
val_stream_tokens = tok.build_scoring_stream(dfs['validation']['dialog'].tolist(), vocab)
test_stream_tokens = tok.build_scoring_stream(dfs['test']['dialog'].tolist(), vocab)

bos_id = vocab.token2id[tok.BOS]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 80

train_ids = vocab.encode(train_stream_tokens)
val_ids = vocab.encode(val_stream_tokens)
test_ids = vocab.encode(test_stream_tokens)
train_data = awd.batchify(train_ids, BATCH_SIZE, device)

print("Train/val/test stream lengths:", len(train_ids), len(val_ids), len(test_ids))
print("Batchified train_data shape:", train_data.shape)

Vocab size: 12482
Train/val/test stream lengths: 1201536 110872 108079
Batchified train_data shape: torch.Size([15019, 80])


In [9]:
try:
    print(model)
except NameError:
    print("`model` is also undefined -- the runtime was reset or disconnected; training state is lost.")

`model` is also undefined -- the runtime was reset or disconnected; training state is lost.


In [ ]:
model = awd.AWDLSTM(
    vocab_size=len(vocab), embed_dim=400, hidden_dim=1150, n_layers=3,
    dropconnect_p=0.5, embed_dropout_p=0.1, output_dropout_p=0.4,
).to(device)
print("Parameter count:", sum(p.numel() for p in model.parameters()))

nt_opt = ntasgd_module.NTASGD(model.parameters(), lr=30.0, n=5, weight_decay=1.2e-6)
criterion = nn.CrossEntropyLoss(ignore_index=bos_id)

def train_one_epoch_ntasgd(model, train_data, nt_opt, criterion, bptt=70, clip=0.25):
    model.train()
    total_loss, total_tokens = 0.0, 0
    hidden = model.init_hidden(BATCH_SIZE, device)
    for i in range(0, train_data.size(0) - 1, bptt):
        data, targets = awd.get_batch(train_data, i, bptt)
        hidden = awd.repackage_hidden(hidden)
        nt_opt.zero_grad()
        logits, hidden = model(data, hidden)
        loss = criterion(logits.view(-1, len(vocab)), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        nt_opt.step_batch()
        valid_count = (targets != bos_id).sum().item()
        total_loss += loss.item() * valid_count
        total_tokens += valid_count
    return math.exp(total_loss / total_tokens)

N_EPOCHS = 50
train_ppl_history, val_ppl_history_raw = [], []

for ep in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_ppl = train_one_epoch_ntasgd(model, train_data, nt_opt, criterion)
    val_ppl_raw = awd.evaluate_stream_exact(model, val_ids, bos_id, device)
    just_triggered = nt_opt.step_epoch(val_ppl_raw)
    if just_triggered:
        print(f"*** NT-ASGD triggered at epoch {ep} ***")
    log_line = f"Epoch {ep}/{N_EPOCHS} | Train PPL: {train_ppl:.2f} | Val PPL (raw): {val_ppl_raw:.2f}"
    if nt_opt.is_averaging and nt_opt._avg_step_count > 0:
        nt_opt.swap_to_averaged(model)
        val_ppl_avg = awd.evaluate_stream_exact(model, val_ids, bos_id, device)
        nt_opt.swap_to_raw(model)
        log_line += f" | Val PPL (averaged): {val_ppl_avg:.2f}"
    print(log_line + f" | Time: {time.time()-t0:.1f}s")
    train_ppl_history.append(train_ppl)
    val_ppl_history_raw.append(val_ppl_raw)

print("\nTraining complete.")

Parameter count: 25216882
Epoch 1/50 | Train PPL: 856.82 | Val PPL (raw): 309.81 | Time: 91.9s
Epoch 2/50 | Train PPL: 242.34 | Val PPL (raw): 138.04 | Time: 106.2s
Epoch 3/50 | Train PPL: 153.82 | Val PPL (raw): 105.09 | Time: 105.8s
Epoch 4/50 | Train PPL: 121.82 | Val PPL (raw): 85.78 | Time: 101.5s
Epoch 5/50 | Train PPL: 103.93 | Val PPL (raw): 69.40 | Time: 98.1s
Epoch 6/50 | Train PPL: 93.00 | Val PPL (raw): 60.79 | Time: 101.6s
Epoch 7/50 | Train PPL: 84.22 | Val PPL (raw): 57.52 | Time: 98.5s
Epoch 8/50 | Train PPL: 79.11 | Val PPL (raw): 53.26 | Time: 97.7s
Epoch 9/50 | Train PPL: 73.74 | Val PPL (raw): 51.52 | Time: 96.9s
Epoch 10/50 | Train PPL: 70.19 | Val PPL (raw): 48.77 | Time: 97.1s
Epoch 11/50 | Train PPL: 67.00 | Val PPL (raw): 46.45 | Time: 96.8s
*** NT-ASGD triggered at epoch 12 ***
Epoch 12/50 | Train PPL: 64.40 | Val PPL (raw): 47.09 | Time: 97.0s
Epoch 13/50 | Train PPL: 61.62 | Val PPL (raw): 44.13 | Val PPL (averaged): 40.67 | Time: 121.6s
Epoch 14/50 | Train 

In [8]:
nt_opt.swap_to_averaged(model)
test_ppl_averaged = awd.evaluate_stream_exact(model, test_ids, bos_id, device)
val_ppl_averaged_final = awd.evaluate_stream_exact(model, val_ids, bos_id, device)
print(f"FINAL RESULTS (averaged weights): Val PPL = {val_ppl_averaged_final:.3f}, Test PPL = {test_ppl_averaged:.3f}")

torch.save(model.state_dict(), "/content/awd_lstm_final_averaged.pt")
print("Saved locally. (This exact file is already hosted at the repository's Releases page for Task 3 to download, so re-uploading isn't necessary for grading purposes.)")

nt_opt.swap_to_raw(model)

NameError: name 'nt_opt' is not defined